# 08 - Black-Litterman Fusion

**Author:** Sacha Huberty

**Purpose:** The heart of the project. Build the equilibrium prior from
asset-class weights, assemble V1 (regime posture), V2 (mean-reversion),
and V3 (technical) into Black-Litterman's (P, Q, Omega), fuse them into
one posterior return vector, and compare it against the prior and
historical means. This replaces every naive posture-switch +
additive-tilt combination used since stage 3
(PROJECT_STRUCTURE.md 5.1: "until here, views can combine naively; BL
formalizes it") with the real fusion the project was always designed
around.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, meanreversion, metrics, regimes, strategy,
    technicals, universe, views,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["black_litterman"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)
returns.tail()

## Analysis / signal logic

### Equilibrium returns from asset-class weights (in-sample snapshot)

`permanent()`'s 25/25/25/25 asset-class split stands in for "the
market portfolio": with an unrestricted, multi-asset-class universe
there is no single true cap-weighted benchmark the way there is for,
say, S&P 500 constituents.

In [ ]:
lookback = cfg["optimization"]["lookback_days"]
snapshot_window = returns.loc[:as_of_universe].tail(lookback)
cov = allocation.covariance_matrix(snapshot_window, method=cfg["optimization"]["covariance"])

market_weights = allocation.permanent(class_bucket)
prior = allocation.equilibrium_returns(market_weights, cov, cfg["black_litterman"]["delta"])
prior.sort_values(ascending=False)

### Building the views: V1 (regime), V2 (mean-reversion), V3 (technical)

V4 (sentiment) is intentionally omitted here: it is fundamentally
live-only (no historical news archive, see sentiment.py and notebook
07), so there is nothing to fuse historically. `views.sentiment_views`
exists and is unit-tested for a live/forward loop.

In [ ]:
market_ticker = cfg["regimes"]["market_ticker"]
hmm_result = regimes.market_regime(returns.loc[:as_of_universe, market_ticker], cfg, posture_cfg)
posture = hmm_result["current_posture"]
print(f"Current HMM posture: {posture}")

meanrev_view = meanreversion.mean_reversion_view(prices.loc[:as_of_universe], cfg)
tech_view = technicals.technical_view(prices.loc[:as_of_universe], cfg)

view_sets = [
    views.regime_view(posture, class_bucket, prior, cfg),
    views.meanreversion_views(meanrev_view, prior, cfg),
    views.technical_views(tech_view, prior, cfg),
]
P, Q, Omega, labels = views.assemble(view_sets, prior.index, cov, cfg)

view_table = pd.DataFrame(
    {"label": labels, "Q": Q, "omega": np.diag(Omega)}
).set_index("label")
view_table

### Posterior returns: BL vs prior vs historical mean

The core Black-Litterman promise: views should nudge the prior, not
replace it wholesale, and disagreement between views and history
should resolve gracefully rather than produce extreme numbers.

In [ ]:
mu_bl = allocation.black_litterman(prior, cov, P, Q, Omega, cfg["black_litterman"]["tau"])
historical_mean = allocation.mean_returns(snapshot_window)

comparison = pd.DataFrame({
    "prior (equilibrium)": prior,
    "posterior (BL)": mu_bl,
    "historical mean": historical_mean,
})
comparison.sort_values("posterior (BL)", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
comparison.sort_values("posterior (BL)").plot(kind="barh", ax=ax, width=0.8)
ax.set_xlabel("Annualized expected return")
ax.set_title("Equilibrium prior vs. BL posterior vs. historical mean")
plt.tight_layout()
plt.show()

### Sensitivity: final weights to each view family

Toggle each view family off one at a time (an early look at what the
ablation study in notebook 11 will formalize) and compare the
resulting max_sharpe(mu_BL, Sigma) weights.

In [ ]:
def weights_with_views(active_view_sets):
    P_, Q_, Omega_, _ = views.assemble(active_view_sets, prior.index, cov, cfg)
    mu = allocation.black_litterman(prior, cov, P_, Q_, Omega_, cfg["black_litterman"]["tau"])
    return allocation.max_sharpe(mu, cov, cfg)


regime_only = weights_with_views([view_sets[0]])
meanrev_only = weights_with_views([view_sets[1]])
technical_only = weights_with_views([view_sets[2]])
all_views = weights_with_views(view_sets)
no_views = weights_with_views([])

sensitivity = pd.DataFrame({
    "no_views (prior only)": no_views,
    "regime_only": regime_only,
    "meanrev_only": meanrev_only,
    "technical_only": technical_only,
    "all_views": all_views,
})
sensitivity

### V1-V3 fused -> backtest OOS

Same buffered-by-`optimization.lookback_days`, reduced-refit-frequency
scope trade as stages 6-7 (autoencoder refits are the expensive part).

In [ ]:
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

backtest_cfg = copy.deepcopy(cfg)
backtest_cfg["anomaly"]["epochs"] = 10
backtest_cfg["anomaly"]["patience"] = 3
backtest_cfg["anomaly"]["refit_frequency_days"] = 126

bl_fn = strategy.black_litterman_strategy(class_bucket, backtest_cfg, posture_cfg)
bl_anomaly_fn = strategy.with_anomaly_override(bl_fn, backtest_cfg)

bl_result_bt = backtest.run(bl_anomaly_fn, backtest_returns, backtest_cfg)

In [ ]:
# Stage 2 and stage 7 (the prior "current best", naive combination)
# baselines, recomputed here (same buffered range) for a direct
# comparison.
def make_classical_strategy(method):
    cov_method = cfg["optimization"]["covariance"]

    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


v1_fn = strategy.regime_switching_strategy(class_bucket, backtest_cfg, posture_cfg)
v1_anomaly_fn = strategy.with_anomaly_override(v1_fn, backtest_cfg)
v1_anomaly_meanrev_fn = strategy.with_meanreversion_tilt(v1_anomaly_fn, backtest_cfg)
naive_combo_fn = strategy.with_technical_view(v1_anomaly_meanrev_fn, backtest_cfg)
phase_flags_fn = strategy.technical_phase_flags(backtest_cfg)

baseline_fns = {
    "permanent": permanent_strategy,
    "risk_parity": make_classical_strategy("risk_parity"),
}
baseline_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in baseline_fns.items()
}
naive_combo_result = backtest.run(
    naive_combo_fn, backtest_returns, backtest_cfg, phase_flags_fn=phase_flags_fn
)
all_results = {
    "black_litterman": bl_result_bt,
    "naive_combo_v1_v2_v3 (stage 7)": naive_combo_result,
    **baseline_results,
}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison_table = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison_table.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "black_litterman" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: Black-Litterman fusion vs. naive combination vs. baselines")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

## Notes / next steps

- **Honest finding, and a genuinely positive one this time:** formalizing the fusion actually closed real ground. Black-Litterman scored OOS Sharpe 0.895 vs. the naive combination's 0.738 (stage 7) and risk_parity's 0.837 -- and BL's max drawdown (-10.5%) was the BEST of all four strategies compared, including permanent (-12.6%). It still trails permanent's Sharpe (1.098), so the honest-reporting discipline continues, but this is the first stage since 3 where adding structure clearly earned its complexity rather than merely not hurting. The utility-gated GMV/Risk-Parity fallback appears to be doing real work here: BL's annualized vol (5.6%) is noticeably lower than every other book's, consistent with the gate sometimes preferring the defensive book over max_sharpe(mu_BL).

- V1 changed roles fundamentally in this stage: it used to SWITCH
  which classical book ran (posture-conditional book selection);
  now it's an asset-class-level VIEW fused alongside V2/V3, and a
  single max_sharpe(mu_BL) book (compared by utility against GMV/Risk
  Parity) is used every week regardless of posture. Both are valid
  designs -- this one is what PROJECT_STRUCTURE.md's weekly pipeline
  (STEP 2-4) actually specifies.
- The stage 3-7 wrapper functions (`with_meanreversion_tilt`,
  `with_technical_view`, `with_sentiment_view`, `regime_switching_
  strategy`) remain in the codebase, tested and functional -- not
  deleted, just superseded as the "current" strategy. They're the
  honest historical record of the naive-combination era.
- V4 (sentiment) is still omitted from the backtest: fundamentally
  live-only, same as stage 6's options positioning. `views.
  sentiment_views` and `with_sentiment_view` exist for a live/forward
  loop that re-scrapes and re-fuses each real week.
- Next (stage 9): full walk-forward validation, hyperparameter grid
  search (IS only, selected for stability not peak Sharpe), and
  turnover tuning -- the first stage where this backtest becomes the
  actual, final, once-only OOS run rather than a per-stage
  demonstration.